# 🎬 CrewAI ➔ Colab ➔ Google Drive 배치 비디오 렌더러

이 노트북은 GitHub에 저장된 `scenes.json`을 가져와 **AnimateDiff GPU 가속**을 통해 연속 비디오를 생성하고, **Google Drive**에 자동으로 저장합니다.

**필수 확인:** 상단 메뉴에서 `런타임` ➔ `런타임 유형 변경` ➔ `T4 GPU` (또는 A100/V100)로 설정되어 있는지 확인하세요.

### 1단계: Google Drive 마운트
생성된 비디오가 안전하게 보관될 Google Drive 폴더(`/content/drive/MyDrive/ai_videos`)를 연결합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!mkdir -p /content/drive/MyDrive/ai_videos
print("✓ Google Drive 마운트 완료!")

### 2단계: GPU 확인 및 의존성 패키지 설치

In [ ]:
!nvidia-smi
!pip install -q diffusers transformers accelerate torch torchvision imageio[ffmpeg]

### 3단계: GitHub 저장소 클론 또는 최신 씬(scenes.json) 동기화
아래 `REPO_URL`에 본인의 GitHub 저장소 주소를 입력하세요.

In [ ]:
import os

# 본인의 GitHub 저장소 URL로 변경하세요
REPO_URL = "https://github.com/YOUR_USERNAME/CrewAI_Colab.git"
PROJECT_DIR = "/content/CrewAI_Colab"

if not os.path.exists(PROJECT_DIR):
    !git clone {REPO_URL} {PROJECT_DIR}
    %cd {PROJECT_DIR}
else:
    %cd {PROJECT_DIR}
    !git pull origin main

print("✓ 저장소 동기화 완료!")

### 4단계: `scenes.json` 프롬프트 확인

In [ ]:
import json

with open("scenes.json", "r", encoding="utf-8") as f:
    scenes = json.load(f)

print(f"총 {len(scenes)}개 씬 로드됨:\n")
for s in scenes:
    print(f"[씬 {s['scene_id']}] {s['file_name']}")
    print(f"프롬프트: {s['prompt']}\n")

### 5단계: 배치 렌더링 시작 🚀
모든 씬을 AnimateDiff로 순차 렌더링하여 Google Drive에 저장합니다.

In [ ]:
!python batch_generator.py --scenes_file scenes.json --output_dir /content/drive/MyDrive/ai_videos

### 6단계: 생성된 비디오 미리보기

In [ ]:
from IPython.display import Video, display
import glob

video_files = sorted(glob.glob("/content/drive/MyDrive/ai_videos/*.mp4"))
if video_files:
    latest_video = video_files[-1]
    print(f"최신 생성 비디오: {latest_video}")
    display(Video(latest_video, embed=True, width=512, height=512))
else:
    print("비디오 파일을 찾을 수 없습니다.")